# ML-04 — Search Intelligence Data Contract

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

Unit of analysis: one row = one content_id (one content page), for the month 2026-03.
Grain source: fact_content_daily_performance is content-day; we aggregate days 1–15 → feature row per page. Days 16–31 → label per page.
Table(s): fact_content_daily_performance (joined with dim_content for static attributes like word_count, content_type).
Time window: March 2026, split feature (day 1–15) / label (day 16–31), no overlap.

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

| Field | Bucket | Why |
|-------|--------|-----|
| `content_id` | Context | join/group key, not signal |
| `client_id` | Context | for client-holdout split + history filter |
| `gsc_data_start`, `ga4_data_start` | Context | filter clients with history before 2026-03 |
| `ga4_data_available` | Context | filter rows where GA4 is FALSE → zero-filled (not real) |
| `impressions_90d`, `ctr`, `avg_position` (days 1–15 agg) | Feature | measured before label window |
| `content_age_days`, `word_count`, `content_type` | Feature | static/pre-existing, knowable anytime |
| `trend_direction` (days 16–31) | Label source | IS what is_declining_label is derived from |
| `trend_pct` | Excluded | leaks the label — same as notebook 02 trap |
| `is_declining_label` | Label | target (declining = 1, not = 0) |

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [21]:
import pandas as pd
from huggingface_hub import hf_hub_download

# Option 1: Use the sample (what you already got)
path = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    filename="fact_content_daily_performance_sample.parquet",
    repo_type="dataset"
)

# Option 2: Try the query table (90-day aggregated)
path = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    filename="fact_content_query_90d.parquet",
    repo_type="dataset"
)

df = pd.read_parquet(path)
print(df)

fact_content_query_90d.parquet: reconstructing file:   0%|          |  0.00B / 60.7MB            

fact_content_query_90d.parquet: downloading bytes:           |  0.00B            

                  client_hash_id           content_hash_id  \
0        client_08a6a72ff48e62c0  content_447894f2faf0d2bc   
1        client_08a6a72ff48e62c0  content_447894f2faf0d2bc   
2        client_08a6a72ff48e62c0  content_447894f2faf0d2bc   
3        client_08a6a72ff48e62c0  content_447894f2faf0d2bc   
4        client_08a6a72ff48e62c0  content_447894f2faf0d2bc   
...                          ...                       ...   
2414243  client_e547b89c05043229  content_17608f489483f8f8   
2414244  client_e547b89c05043229  content_17608f489483f8f8   
2414245  client_e547b89c05043229  content_17608f489483f8f8   
2414246  client_e547b89c05043229  content_17608f489483f8f8   
2414247  client_e547b89c05043229  content_17608f489483f8f8   

                  query_hash_id  query_char_count  query_token_count  \
0        query_58b1b001f839d699                17                  3   
1        query_922b8eca2a24cd34                34                  7   
2        query_9f0c36a6ae2a6a99        

In [ ]:
import pandas as pd

# Use the CORRECT column names from the sample
df = pd.read_parquet(
    'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance_sample.parquet',
    columns=['report_date', 'client_hash_id', 'gsc_impressions', 'gsc_clicks', 'gsc_avg_position']
)

# Filter for June data
df['report_date'] = pd.to_datetime(df['report_date'])
june_data = df[(df['report_date'] >= '2026-06-01') & (df['report_date'] < '2026-07-01')]
print(june_data.head())
print(f"Shape: {june_data.shape}")

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.